# ClubElo promoted-club prior workbench (prototype)

**Purpose**: construct `features/priors.py.build_prior()`'s `external_rating` from
ClubElo (the narrower, already-designed use case from ADR 0006 -- not a competing
match model, unlike `03-02`) and check whether it actually improves the promoted-club
prior, before any production wiring.

**A real, already-documented gap this notebook fills itself**: the promoted-club prior
is not exercised by `evaluate/backtest.py`'s walk-forward harness at all --
`run_backtest` just records a club with no history as `unrateable` and moves on. Only
`forecasting.fit_and_simulate` (the live path) ever calls the prior machinery. So there
is no existing RPS number for "does the prior help," ClubElo-backed or not, before this
notebook. Section 2 below builds a dedicated evaluation harness, reusing
`needs_prior`/`build_survival_zone_reference`/`build_prior`/`prior_pseudo_matches` --
all public, already tested, in `features/priors.py` -- rather than a
plausibility-only check.

**Small-sample caveat, stated up front**: ClubElo's cached history (`03-02`) only
covers roughly 2022/23 onward, so the number of real historical promoted-club debuts
this can evaluate is small. Treat the numbers below as suggestive, not a large
backtest.

Standard prototype, not a standing exception (same footing as `03-02`/`03-03`): stays,
moves into `src/plforecast`, or gets deleted once its question is answered. Nothing
here is wired into `forecasting.py` or `club_aliases.yaml` -- that is a separate
decision once this notebook's evidence is in.


In [ ]:
import json
from datetime import date
from pathlib import Path

import numpy as np
import polars as pl

from plforecast import forecasting, viz
from plforecast.artifacts.schema import Provenance
from plforecast.artifacts.writer import (
    build_forecast_document,
    git_provenance,
    now_utc,
    package_versions,
    snapshot_provenance,
)
from plforecast.config import Settings
from plforecast.entities.clubs import load_club_dimension
from plforecast.evaluate.metrics import (
    brier_score,
    log_loss,
    outcome_index,
    outcome_probabilities,
    rps,
)
from plforecast.evaluate.report import format_table
from plforecast.features.priors import (
    PRIOR_GATE_XI,
    build_prior,
    build_survival_zone_reference,
    needs_prior,
    prior_pseudo_matches,
)
from plforecast.features.strength import build_club_strength
from plforecast.models.base import UnknownClubError, drop_odds_columns
from plforecast.simulate.competition import PREMIER_LEAGUE
from plforecast.simulate.engine import simulate_season
from plforecast.simulate.tiebreak import PremierLeagueTiebreaks
from plforecast.storage.db import connect

REPO_ROOT = Path("../..")
NOTEBOOK_CONFIG = Settings(data_dir=REPO_ROOT / "data")
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
CLUBELO_CACHE_DIR = REPO_ROOT / "data" / "clubelo-cache"
COMPARISON_SEASON_FLOOR = "2022/23"  # ClubElo's available history starts ~Sept 2022 (03-02)

In [ ]:
def load_elo_history() -> dict[str, pl.DataFrame]:
    frames = {}
    for path in sorted(CLUBELO_CACHE_DIR.glob("*.json")):
        rows = json.loads(path.read_text())
        if not rows:
            continue
        frames[path.stem] = (
            pl.DataFrame(rows)
            .select(
                pl.col("Date").str.slice(0, 10).str.to_date().alias("date"),
                pl.col("Elo").alias("elo"),
            )
            .sort("date")
        )
    return frames


elo_history = load_elo_history()
if not elo_history:
    raise RuntimeError(
        "no cached ClubElo data found -- run 03-02-clubelo-workbench.ipynb's section 1 first"
    )
print(f"loaded Elo history for {len(elo_history)} clubs")

In [ ]:
conn = connect(NOTEBOOK_CONFIG)
evaluation_matches = conn.execute(
    "SELECT match_id, season, date, home_club_id, away_club_id, home_goals, away_goals, "
    "home_xg, away_xg, result FROM stg_matches "
    "WHERE season != (SELECT max(season) FROM stg_matches) ORDER BY date"
).pl()
conn.close()
print(f"{evaluation_matches.height} matches, {evaluation_matches['season'].n_unique()} seasons")

## Section 1: Elo-to-rate calibration

`build_prior()`'s `external_rating` needs goals-per-match rates (`home_attack`,
`home_defence`, `away_attack`, `away_defence`), not Elo points. Bridges the two with
four small log-linear regressions, `log(rate) = a + b * elo`, fit on *established*
clubs where both a `features/strength.py` rate snapshot and a ClubElo rating exist at
the same `as_of` date -- the same log-space regression shape `03-02`/`03-03` already
used. Refit at every point it's used below (never fit once and applied backward), so
nothing here has lookahead bias.

In [ ]:
RATE_FIELDS = ["home_attack_rate", "home_defence_rate", "away_attack_rate", "away_defence_rate"]
EXTERNAL_KEYS = {
    "home_attack_rate": "home_attack",
    "home_defence_rate": "home_defence",
    "away_attack_rate": "away_attack",
    "away_defence_rate": "away_defence",
}


def fit_elo_to_rate(
    matches: pl.DataFrame,
    elo_history: dict[str, pl.DataFrame],
    *,
    as_of: date,
    strength_xi: float = 0.0018,
) -> tuple[dict[str, tuple[float, float]], pl.DataFrame]:
    """(slope, intercept) per rate field, fit on established clubs with both signals
    as of `as_of`. Returns the calibration frame too, for a sanity-check plot."""
    strength = build_club_strength(matches, as_of=as_of, xi=strength_xi)
    rows = []
    for row in strength.iter_rows(named=True):
        history = elo_history.get(row["club_id"])
        if history is None:
            continue
        at_or_before = history.filter(pl.col("date") <= as_of)
        if at_or_before.height == 0:
            continue
        rows.append({**row, "elo": float(at_or_before[-1, "elo"])})
    schema = {"club_id": pl.Utf8, **{f: pl.Float64 for f in RATE_FIELDS}, "elo": pl.Float64}
    calib = pl.DataFrame(rows, schema=schema).drop_nulls(RATE_FIELDS)
    coefficients: dict[str, tuple[float, float]] = {}
    # Too early in the backfill window for ClubElo's cached history to have started
    # yet (03-02: coverage begins ~Sept 2022, mid-way into the 2022/23 season) --
    # calibration is empty for that case, not an error; callers fall back to the
    # survival-zone anchor alone, same as a club with no external rating at all.
    if calib.height >= 2:
        for field in RATE_FIELDS:
            values = calib[field].to_numpy()
            elo_vals = calib["elo"].to_numpy()
            valid = values > 0
            if valid.sum() >= 2:
                slope, intercept = np.polyfit(elo_vals[valid], np.log(values[valid]), 1)
                coefficients[field] = (float(slope), float(intercept))
    return coefficients, calib


def predict_external_rating(
    coefficients: dict[str, tuple[float, float]], elo: float
) -> dict[str, float]:
    return {
        EXTERNAL_KEYS[field]: float(np.exp(intercept + slope * elo))
        for field, (slope, intercept) in coefficients.items()
    }

Sanity check: fitted vs. actual home attack rate for established clubs, today's snapshot.

In [ ]:
import plotly.express as px

_coefficients_today, _calib_today = fit_elo_to_rate(
    evaluation_matches, elo_history, as_of=date.today()
)
_slope, _intercept = _coefficients_today["home_attack_rate"]
_calib_today = _calib_today.with_columns(
    (np.exp(_intercept + _slope * pl.col("elo"))).alias("fitted_home_attack_rate")
)
fig = px.scatter(
    _calib_today.to_pandas(),
    x="elo",
    y="home_attack_rate",
    hover_name="club_id",
    title="Home attack rate vs. ClubElo (points = actual, line = fitted)",
)
fig.add_scatter(
    x=_calib_today.sort("elo")["elo"],
    y=_calib_today.sort("elo")["fitted_home_attack_rate"],
    mode="lines",
    name="fitted",
)
fig.show()

## Section 2: evaluation harness

For every ClubElo-covered season, finds every club that trips `needs_prior()` at that
season's start (the same gate `forecasting._inject_promoted_club_prior` uses live),
takes each flagged club's first `DEBUT_WINDOW_MATCHES` matches that season as the
evaluation rows, and prices each one two ways: today's shipped behaviour (survival-zone
anchor alone) and ClubElo-blended at `EXTERNAL_WEIGHT`. Reuses `forecasting
.build_model_factory("xg-rates")` for the match model, so this mirrors the shipped
model's own tuned hyperparameters rather than re-guessing them.

In [ ]:
# ---- parameters: edit and re-run this cell and everything below it ----
EXTERNAL_WEIGHT = 0.5
DEBUT_WINDOW_MATCHES = 6

In [ ]:
def _season_start(matches: pl.DataFrame, season: str) -> date:
    return matches.filter(pl.col("season") == season)["date"].min()


def evaluate_prior(
    full_history: pl.DataFrame,
    elo_history: dict[str, pl.DataFrame],
    *,
    external_weight: float,
    seasons: list[str],
    debut_window: int = DEBUT_WINDOW_MATCHES,
    seed: int = 0,
) -> pl.DataFrame:
    rows = []
    for season in seasons:
        as_of = _season_start(full_history, season)
        completed = full_history.filter(pl.col("date") < as_of)
        season_matches = full_history.filter(pl.col("season") == season).sort("date")
        fixture_clubs = sorted(
            set(season_matches["home_club_id"]) | set(season_matches["away_club_id"])
        )
        needing = [
            c for c in fixture_clubs if needs_prior(c, completed, as_of=as_of, xi=PRIOR_GATE_XI)
        ]
        if not needing:
            continue

        reference = build_survival_zone_reference(completed)
        coefficients, _ = fit_elo_to_rate(completed, elo_history, as_of=as_of)

        target_match_ids: set[str] = set()
        for club in needing:
            club_matches = season_matches.filter(
                (pl.col("home_club_id") == club) | (pl.col("away_club_id") == club)
            ).head(debut_window)
            target_match_ids.update(club_matches["match_id"].to_list())
        target_matches = season_matches.filter(pl.col("match_id").is_in(target_match_ids)).sort(
            "date"
        )

        for row in target_matches.iter_rows(named=True):
            train = pl.concat([completed, season_matches.filter(pl.col("date") < row["date"])])
            pseudo_frames = []
            for club in needing:
                external_rating = None
                if external_weight > 0:
                    history = elo_history.get(club)
                    if history is not None:
                        elo_row = history.filter(pl.col("date") <= as_of)
                        if elo_row.height:
                            external_rating = predict_external_rating(
                                coefficients, float(elo_row[-1, "elo"])
                            )
                prior = build_prior(
                    club,
                    reference,
                    external_rating=external_rating,
                    external_weight=external_weight if external_rating else 0.0,
                )
                pseudo_frames.append(
                    prior_pseudo_matches(
                        prior, opponents=fixture_clubs, season=season, as_of=as_of, seed=seed
                    )
                )
            augmented = pl.concat([train, *pseudo_frames], how="vertical_relaxed")
            try:
                fitted = forecasting.build_model_factory("xg-rates")().fit(
                    drop_odds_columns(augmented)
                )
                matrix = fitted.scoreline_matrix(
                    row["home_club_id"], row["away_club_id"], max_goals=10
                )
            except UnknownClubError:
                continue
            probs = outcome_probabilities(matrix)
            rows.append(
                {
                    "season": season,
                    "date": row["date"],
                    "home_club_id": row["home_club_id"],
                    "away_club_id": row["away_club_id"],
                    "result": row["result"],
                    "p_home": probs[0],
                    "p_draw": probs[1],
                    "p_away": probs[2],
                }
            )

    schema = {
        "season": pl.Utf8,
        "date": pl.Date,
        "home_club_id": pl.Utf8,
        "away_club_id": pl.Utf8,
        "result": pl.Utf8,
        "p_home": pl.Float64,
        "p_draw": pl.Float64,
        "p_away": pl.Float64,
    }
    scored = pl.DataFrame(rows, schema=schema)
    if scored.height == 0:
        return scored
    probs = scored.select("p_home", "p_draw", "p_away").to_numpy()
    outcomes = outcome_index(scored["result"].to_list())
    return scored.with_columns(
        pl.Series("rps", rps(probs, outcomes)),
        pl.Series("log_loss", log_loss(probs, outcomes)),
        pl.Series("brier", brier_score(probs, outcomes)),
    )

In [ ]:
SEASONS = sorted(
    s for s in evaluation_matches["season"].unique().to_list() if s >= COMPARISON_SEASON_FLOOR
)
print(f"evaluating seasons: {SEASONS}")

survival_zone_scores = evaluate_prior(
    evaluation_matches, elo_history, external_weight=0.0, seasons=SEASONS
)
clubelo_scores = evaluate_prior(
    evaluation_matches, elo_history, external_weight=EXTERNAL_WEIGHT, seasons=SEASONS
)

print(
    f"evaluation rows: survival-zone-only {survival_zone_scores.height}, "
    f"clubelo-blended {clubelo_scores.height}"
)
if survival_zone_scores.height and clubelo_scores.height:
    print(
        format_table(
            [
                {
                    "model": "survival-zone-only",
                    "n": survival_zone_scores.height,
                    "rps": survival_zone_scores["rps"].mean(),
                    "log_loss": survival_zone_scores["log_loss"].mean(),
                    "brier": survival_zone_scores["brier"].mean(),
                },
                {
                    "model": "clubelo-blended",
                    "n": clubelo_scores.height,
                    "rps": clubelo_scores["rps"].mean(),
                    "log_loss": clubelo_scores["log_loss"].mean(),
                    "brier": clubelo_scores["brier"].mean(),
                },
            ]
        )
    )
else:
    print("too few evaluation rows in this window to compare -- see the small-sample caveat above")

Which specific debut cases are in the evaluation set:

In [ ]:
survival_zone_scores.select("season", "home_club_id", "away_club_id", "date").sort("season", "date")

### Sweep `EXTERNAL_WEIGHT`

Does trusting ClubElo more actually help, or just narrow the prior on noise?

In [ ]:
GRID = [0.0, 0.25, 0.5, 0.75, 1.0]
sweep_rows = []
for weight in GRID:
    scores = evaluate_prior(
        evaluation_matches, elo_history, external_weight=weight, seasons=SEASONS
    )
    sweep_rows.append(
        {
            "value": weight,
            "n": scores.height,
            "rps": scores["rps"].mean() if scores.height else float("nan"),
        }
    )
sweep_df = pl.DataFrame(sweep_rows)
print(sweep_df)
viz.lever_comparison(sweep_rows, metric="rps", x_key="value")

## Section 3: forecast visualization

A live forecast from `XGRateModel` with the ClubElo-blended prior applied to this
season's actual promoted clubs -- the final projected table below is the headline
output, same as `03-01`/`03-02`/`03-03`. Then a comparison against the current shipped
forecast (`forecasting.fit_and_simulate("xg-rates", ...)`, unmodified), since the
question here is the prior specifically, not a whole competing model.

In [ ]:
SIMULATIONS = 5_000
SEED = 42

baseline_run = forecasting.fit_and_simulate(
    "xg-rates", simulations=SIMULATIONS, seed=SEED, config=NOTEBOOK_CONFIG
)
print(
    f"{baseline_run.season_label}, as of gameweek {baseline_run.as_of_gameweek}, "
    f"promoted-club prior applied to: {baseline_run.promoted_clubs}"
)

In [ ]:
conn = connect(NOTEBOOK_CONFIG)
current_fixtures = conn.execute(
    "SELECT fixture_id, season, gameweek, kickoff_time, home_club_id, away_club_id, "
    "home_goals, away_goals, finished FROM stg_fixtures ORDER BY kickoff_time, fixture_id"
).pl()
full_current_history = conn.execute(
    "SELECT match_id, season, date, home_club_id, away_club_id, home_goals, away_goals, "
    "home_xg, away_xg, result FROM stg_matches ORDER BY date"
).pl()
conn.close()

season_label = str(current_fixtures["season"][0])
as_of_live = date.today()
played = current_fixtures.filter(pl.col("finished"))
remaining = current_fixtures.filter(~pl.col("finished"))
completed_for_live = full_current_history.filter(pl.col("season") != season_label)
current_season_matches = full_current_history.filter(pl.col("season") == season_label)
fixture_clubs_live = sorted(
    set(current_fixtures["home_club_id"]) | set(current_fixtures["away_club_id"])
)
needing_live = [
    c
    for c in fixture_clubs_live
    if needs_prior(c, completed_for_live, as_of=as_of_live, xi=PRIOR_GATE_XI)
]
print(f"clubs needing a prior this season: {needing_live}")

reference_live = build_survival_zone_reference(completed_for_live)
coefficients_live, _ = fit_elo_to_rate(completed_for_live, elo_history, as_of=as_of_live)

In [ ]:
def build_clubelo_training(external_weight: float) -> pl.DataFrame:
    train = pl.concat([completed_for_live, current_season_matches])
    pseudo_frames = []
    for club in needing_live:
        external_rating = None
        if external_weight > 0:
            history = elo_history.get(club)
            if history is not None:
                elo_row = history.filter(pl.col("date") <= as_of_live)
                if elo_row.height:
                    external_rating = predict_external_rating(
                        coefficients_live, float(elo_row[-1, "elo"])
                    )
        prior = build_prior(
            club,
            reference_live,
            external_rating=external_rating,
            external_weight=external_weight if external_rating else 0.0,
        )
        pseudo_frames.append(
            prior_pseudo_matches(
                prior,
                opponents=fixture_clubs_live,
                season=season_label,
                as_of=as_of_live,
                seed=SEED,
            )
        )
    return pl.concat([train, *pseudo_frames], how="vertical_relaxed") if pseudo_frames else train


clubelo_training = build_clubelo_training(EXTERNAL_WEIGHT)
clubelo_fitted = forecasting.build_model_factory("xg-rates")().fit(
    drop_odds_columns(clubelo_training)
)

clubelo_result = simulate_season(
    played.select("home_club_id", "away_club_id", "home_goals", "away_goals"),
    remaining.select("home_club_id", "away_club_id"),
    clubelo_fitted,
    PremierLeagueTiebreaks(PREMIER_LEAGUE),
    n_simulations=SIMULATIONS,
    max_goals=10,
    seed=SEED,
)
assert (clubelo_result.position_counts.sum(axis=1) == SIMULATIONS).all()
assert (clubelo_result.position_counts.sum(axis=0) == SIMULATIONS).all()
print(f"simulated {len(clubelo_result.club_ids)} clubs with the ClubElo-blended prior")

### Final projected table (ClubElo-blended prior)

In [ ]:
dimension = load_club_dimension()
display_names = dict(
    zip(
        dimension.frame["club_id"].to_list(), dimension.frame["display_name"].to_list(), strict=True
    )
)


def _build_document(result, played_df):
    generated_at = now_utc()
    sha, dirty = git_provenance(REPO_ROOT)
    provenance = Provenance(
        git_sha=sha,
        git_dirty=dirty,
        model="xg-rates",
        model_config_hash=clubelo_fitted.config_hash,
        simulations=SIMULATIONS,
        random_seed=SEED,
        sources=snapshot_provenance(NOTEBOOK_CONFIG.raw_dir),
        package_versions=package_versions(),
    )
    return build_forecast_document(
        result,
        played=played_df,
        display_names=display_names,
        season=season_label.replace("/", "-"),
        as_of_gameweek=baseline_run.as_of_gameweek,
        generated_at=generated_at,
        provenance=provenance,
    )


baseline_doc = _build_document(baseline_run.result, baseline_run.played)
clubelo_doc = _build_document(clubelo_result, played)

clubelo_final_table = pl.DataFrame(
    [
        {
            "pos": i + 1,
            "club": club.display_name,
            "pld": club.current.played,
            "pts": club.current.points,
            "E[pts]": round(club.projected.points.mean, 1),
            "title%": round(club.projected.title * 100, 1),
            "top4%": round(club.projected.top_four * 100, 1),
            "rel%": round(club.projected.relegation * 100, 1),
        }
        for i, club in enumerate(clubelo_doc.clubs)
    ]
)
clubelo_final_table

In [ ]:
viz.position_matrix(clubelo_result.club_ids, clubelo_result.position_pmf, display_names)

In [ ]:
viz.points_ridgeline(clubelo_result.club_ids, clubelo_result.points, display_names)

### Comparison: shipped prior vs. ClubElo-blended prior

In [ ]:
baseline_by_club = {c.club_id: c for c in baseline_doc.clubs}
clubelo_by_club = {c.club_id: c for c in clubelo_doc.clubs}

comparison_rows = []
for club_id in set(baseline_by_club) | set(clubelo_by_club):
    b = baseline_by_club.get(club_id)
    e = clubelo_by_club.get(club_id)
    comparison_rows.append(
        {
            "club": display_names.get(club_id, club_id),
            "needed_prior": club_id in needing_live,
            "shipped_e_pts": round(b.projected.points.mean, 1) if b else None,
            "clubelo_e_pts": round(e.projected.points.mean, 1) if e else None,
            "shipped_rel%": round(b.projected.relegation * 100, 1) if b else None,
            "clubelo_rel%": round(e.projected.relegation * 100, 1) if e else None,
        }
    )
comparison = pl.DataFrame(comparison_rows).sort("needed_prior", descending=True)
comparison